# Port Data Catalog Updater

In [30]:
import pandas as pd

# ======================================================
# CONFIG — CHANGE THESE EACH YEAR
# ======================================================
REPORTING_YEAR = 2021

EXCEL_FILE = f"PortPerformance{REPORTING_YEAR}.xlsx"
HISTORICAL_FILE = "Port_Data_20251112.csv"
OUTPUT_FILE = f"Annual_Port_Statistics_{REPORTING_YEAR}_UPDATE.csv"

# ======================================================
# LOAD WORKBOOK + HISTORICAL DATA
# ======================================================
xls = pd.ExcelFile(EXCEL_FILE)
df_hist = pd.read_csv(HISTORICAL_FILE)
df_hist['Port ID'] = df_hist['Port ID'].str.replace(",","").astype(int)

df_hist['Percent Change'] = df_hist['Percent Change'].str.replace(",","").astype(float) 
df_hist['Percent Change'] = df_hist['Percent Change'].round(1) 
df_hist_year = df_hist[df_hist["Reporting Year"] == REPORTING_YEAR]
df_hist_year_minus_1 = df_hist[df_hist["Reporting Year"] == REPORTING_YEAR - 1]
#df_hist_year_minus_1['Port ID'] = df_hist_year_minus_1['Port ID'].str.replace(",","").astype(int)
df_hist_year_minus_1['Volume'] = df_hist_year_minus_1['Volume'].str.replace(",","").astype(float)
records = []

# ======================================================
# RECONSTRUCT AGRICULTURAL COMMODITY LIST (AUTHORITATIVE)
# ======================================================


/tmp/ipykernel_169299/3541824184.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hist_year_minus_1['Volume'] = df_hist_year_minus_1['Volume'].str.replace(",","").astype(float)


## Helper Functions

### Get State

In [2]:
tmp = xls.parse("Ports by ICST")
# df['State'] = df['PORT_NAME'].str.split(', ').str[-1]
# df.head()
# state_abbrev_to_name = {
#     'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
#     'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
#     'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
#     'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
#     'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
#     'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
#     'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
#     'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
#     'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
#     'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
#     'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
#     'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
#     'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia',
#     'PR': 'Puerto Rico', 'VI': 'Virgin Islands', 'GU': 'Guam', 'AS': 'American Samoa'
# }

# Usage
#df['State_Full'] = df['State'].map(state_abbrev_to_name)
tmp.head()


,PORT,PORT_NAME,CATEGORY,INBOUND,OUTBOUND,TOTAL
0,77,"Searsport, ME",Dry Bulk,7,6,13
1,77,"Searsport, ME",Other Freight Barge,21,20,41
2,77,"Searsport, ME",Other Freight,24,27,51
3,78,"Portsmouth, NH",Other Freight Barge,7,7,14
4,78,"Portsmouth, NH",Dry Bulk,13,13,26


### Get Station List form Historical Data

In [3]:
ports_hist = df_hist[['Port_Name','Port ID','State', 'Region']].drop_duplicates()
#ports_hist['Port ID'] = ports_hist['Port ID'].str.replace(",","")
ports_hist.dropna(inplace=True)



## Total Tonage

In [4]:
df = xls.parse("Top Ports")
print("DF ORIG SHAPE",df.shape)
## Get Port ID from the POrts by ICST sheet
tmp = xls.parse("Ports by ICST")
tmp = tmp[["PORT","PORT_NAME"]].drop_duplicates()
df=pd.merge(df,tmp,how="left",left_on="PORT NAME",right_on="PORT_NAME")
df.drop(columns=["PORT_NAME"],inplace=True)
print("DF NEW SHAPE",df.shape)
# df['PORT'] = df['PORT'].astype(int) 

mapr = {'GRAND TOTAL':'TOTAL','FOREIGN TOTAL':'FOREIGN','IMPORTS':'IMPORTS','EXPORTS':'EXPORTS','DOMESTIC':'DOMESTIC'}
all_records = []
for idx,row in ports_hist.iterrows():
  #  print(row)
    port_id = row['Port ID']
    hld=df.loc[df['PORT'] == port_id]
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOTAL TONNAGE')]
    
    if hld.shape[0]==0:
        print("No data for port ID:", port_id, row['Port_Name'])
        continue
    for tt in ['GRAND TOTAL','FOREIGN TOTAL','IMPORTS','EXPORTS','DOMESTIC']:
        tp = mapr[tt]
        vol_minus_1 = hst[hst['Trade Type'] == tp]['Volume'].values[0] if not hst.empty else None              
       
        if vol_minus_1:
            percent_change = ((hld[tt].values[0] - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tp}. Cannot calculate percent change.")




        tmp = {
          'Cargo Type': 'TOTAL TONNAGE', 
          'Port ID': int(port_id), 
          'Port_Name': row['Port_Name'], 
          'Region': row['Region'],
          'Reporting Year': REPORTING_YEAR,
          'State': row['State'], 
          'Trade Type':tp, 
          'Units':'Short Tons', 
          'Port Ranking': hld['RANK'].values[0],
          'Percent Change':percent_change,
          'Volume': hld[tt].values[0]
        }
       
        all_records.append(tmp)

totalTonnage = pd.DataFrame(all_records)
print("Total Tonnage shape:",totalTonnage.shape)

        

DF ORIG SHAPE (299, 10)
DF NEW SHAPE (299, 11)
No historical volume for port ID Alaska, AK Port of 4820, trade type EXPORTS. Cannot calculate percent change.
No data for port ID: 2338 Cincinnati-Northern KY, Ports of
No data for port ID: 2083 Gulfport, MS
No data for port ID: 2348 Huntington-Tristate, KY, OH, WV
No historical volume for port ID Mid-America Port Commission 2306, trade type FOREIGN. Cannot calculate percent change.
No historical volume for port ID Mid-America Port Commission 2306, trade type IMPORTS. Cannot calculate percent change.
No historical volume for port ID Mid-America Port Commission 2306, trade type EXPORTS. Cannot calculate percent change.
No historical volume for port ID New Bourbon Port, MO 2351, trade type FOREIGN. Cannot calculate percent change.
No historical volume for port ID New Bourbon Port, MO 2351, trade type IMPORTS. Cannot calculate percent change.
No historical volume for port ID New Bourbon Port, MO 2351, trade type EXPORTS. Cannot calculate per

### Total Check

#### Internal Checks

In [41]:
nbad=0
ngood=0
ntotal=0

for pid in totalTonnage['Port ID'].unique():
    row = totalTonnage[totalTonnage['Port ID'] == pid]
    hld={}
    bad=False
    ntotal+=1
    for tt in row['Trade Type'].unique():
        hld[tt] = row[row['Trade Type'] == tt]['Volume'].values[0]
    frgn = hld['FOREIGN']
    dmst = hld['DOMESTIC']
    tot = hld['TOTAL']
    impt = hld['IMPORTS']
    expo = hld['EXPORTS']
    if frgn != impt + expo:
        print(f"Mismatch for Port ID {pid}: FOREIGN {frgn} vs IMPORTS+EXPORTS {impt+expo} IMPT {impt} EXPO {expo}")
        bad=True
    if tot != dmst + frgn:
        print(f"Mismatch for Port ID {pid}: TOTAL {tot} vs DOMESTIC+FOREIGN {dmst+frgn} DMST {dmst} FRGN {frgn}")
        bad=True

    if bad:
        nbad+=1
    else:
        ngood+=1

print(f"Checked {ntotal} ports with {nbad} mismatches and {ngood} good records.")

# Get value counts for the 'Trade Type' column
counts = totalTonnage['Trade Type'].value_counts()

# Check if all counts are equal
all_equal = counts.nunique() == 1
print(f"All types have equal counts: {all_equal}")

# See which types have different counts
print(f"\nMin count: {counts.min()}")
print(f"Max count: {counts.max()}")
print(f"Difference: {counts.max() - counts.min()}")
#    print(f"Port ID {pid}, Total Volume from Trade Types: {tot}, Total Volume from TOTAL row: {row[row['Trade Type'] == 'TOTAL']['Volume'].values[0]}")

Checked 48 ports with 0 mismatches and 48 good records.
All types have equal counts: True

Min count: 48
Max count: 48
Difference: 0


#### Historical Check

In [7]:
tot_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == "TOTAL TONNAGE"].copy()
tot_hist.shape
#tot_hist['Port ID']= tot_hist['Port ID'].str.replace(",","")
tot_hist['Volume']= tot_hist['Volume'].str.replace(",","").astype(int)

tot_hist['Port Ranking']= tot_hist['Port Ranking'].astype(int)

totalTonnage.sort_values(by=["Port ID","Trade Type"],inplace=True)
totalTonnage.reset_index(drop=True,inplace=True)
tot_hist.sort_values(by=["Port ID","Trade Type"],inplace=True)
tot_hist.reset_index(drop=True,inplace=True)
nchecks=0
nbad=0
for idx,row in totalTonnage.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'] 
    nchecks+=1
    print(f"Checking Port ID {prt}, Port Name {row['Port_Name']}, Trade Type {tp}..." )
    hst= tot_hist.loc[(tot_hist['Port ID']==prt) & (tot_hist['Trade Type']==tp)] 
    print(f"Historical record found: {not hst.empty}")     
    for ii in row.index:
        if str(row[ii]).strip() != str(hst[ii].values[0]).strip() :
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}")
             nbad+=1

print(f"Completed {nchecks} checks with {nbad} mismatches.")
totalTonnage_Stns = set(totalTonnage['Port_Name'].unique().tolist())
totalTonnage_hist_Stns = set(tot_hist['Port_Name'].unique().tolist())


print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",totalTonnage.shape)
print("Update Shape",tot_hist.shape)
print("Total Tonnage Ports NEW not in HIST:", totalTonnage_Stns - totalTonnage_hist_Stns)
print("Total Tonnage Ports HIST not in NEW:", totalTonnage_hist_Stns - totalTonnage_Stns)

Checking Port ID 149, Port Name Boston, MA, Trade Type DOMESTIC...
Historical record found: True
Checking Port ID 149, Port Name Boston, MA, Trade Type EXPORTS...
Historical record found: True
Checking Port ID 149, Port Name Boston, MA, Trade Type FOREIGN...
Historical record found: True
Checking Port ID 149, Port Name Boston, MA, Trade Type IMPORTS...
Historical record found: True
Checking Port ID 149, Port Name Boston, MA, Trade Type TOTAL...
Historical record found: True
Checking Port ID 398, Port Name New York, NY & NJ, Trade Type DOMESTIC...
Historical record found: True
Checking Port ID 398, Port Name New York, NY & NJ, Trade Type EXPORTS...
Historical record found: True
Checking Port ID 398, Port Name New York, NY & NJ, Trade Type FOREIGN...
Historical record found: True
Checking Port ID 398, Port Name New York, NY & NJ, Trade Type IMPORTS...
Historical record found: True
Checking Port ID 398, Port Name New York, NY & NJ, Trade Type TOTAL...
Historical record found: True
Checkin

## Dry Bulk

In [ ]:

df = xls.parse("Dry Bulk")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("DRY BULK TONNAGE COLUMNS:", df.columns.tolist())
df['PORT'] = df['PORT'].astype('Int64')
## Get Port ID from the POrts by ICST sheet
# tmp = xls.parse("Ports by ICST")
# tmp = tmp[["PORT","PORT_NAME"]].drop_duplicates()
# df=pd.merge(df,tmp,how="left",left_on="PORT NAME",right_on="PORT_NAME")
# df.drop(columns=["PORT_NAME"],inplace=True)
# print("DF NEW SHAPE",df.shape)

all_records = []
for idx,row in ports_hist.iterrows():
  #  print(row)
    port_id = int(row['Port ID'])
    hld=df.loc[df['PORT'] == port_id]
    if hld.shape[0]==0:
        print("No data for port ID:",port_id,row['Port_Name'])
        continue
    
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'DRY BULK')]

    for tt in ['TOTAL','FOREIGN','IMPORTS','EXPORTS','DOMESTIC']:
        volume=int(hld[tt].values[0]) if pd.notna(hld[tt].values[0]) else 0
        vol_minus_1 = hst[hst['Trade Type'] == tt]['Volume'].values[0] if not hst.empty else None              
       
        if vol_minus_1:
            percent_change = ((hld[tt].values[0] - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")




        tmp = {
          'Cargo Type': 'DRY BULK', 
          'Port ID': port_id, 
          'Port_Name': row['Port_Name'], 
          'Region': row['Region'],
          'Reporting Year': REPORTING_YEAR,
          'State': row['State'], 
          'Trade Type':tt, 
          'Units':'Short Tons', 
          'Port Ranking': hld.index.values[0]+1,
          'Percent Change':percent_change,
          'Volume': volume
        }
       
        all_records.append(tmp)

dryBulk = pd.DataFrame(all_records)
print("Dry Bulk shape:",dryBulk.shape)

        

DRY BULK TONNAGE COLUMNS: ['PORT', 'PORT NAME', 'DOMESTIC', 'EXPORTS', 'IMPORTS', 'FOREIGN', 'TOTAL', 'UNNAMED: 7', 'UNNAMED: 8', 'UNNAMED: 9', 'UNNAMED: 10']
No historical volume for port ID Alaska, AK Port of 4820, trade type EXPORTS. Cannot calculate percent change.
No data for port ID: 2338 Cincinnati-Northern KY, Ports of
No data for port ID: 2083 Gulfport, MS
No data for port ID: 2348 Huntington-Tristate, KY, OH, WV
No historical volume for port ID Port Freeport, TX 2408, trade type EXPORTS. Cannot calculate percent change.
No historical volume for port ID Port of Charleston, SC 775, trade type EXPORTS. Cannot calculate percent change.
No historical volume for port ID Port of Palm Beach District, FL 1983, trade type IMPORTS. Cannot calculate percent change.
No historical volume for port ID PortMiami, FL 1992, trade type EXPORTS. Cannot calculate percent change.
No historical volume for port ID San Juan, PR 1913, trade type EXPORTS. Cannot calculate percent change.
No historical v

In [18]:
df.head()

,PORT,PORT NAME,DOMESTIC,EXPORTS,IMPORTS,FOREIGN,TOTAL,UNNAMED: 7,UNNAMED: 8,UNNAMED: 9,UNNAMED: 10
0,2253,"South Louisiana, LA, Port of",75940355.0,59047191.0,19277065.0,78324256.0,154264611,NaN,1.529763e+08,1288315.0,NaN
1,2251,"New Orleans, LA",29189964.0,15308722.0,9783225.0,25091947.0,54281911,NaN,NaN,NaN,NaN
2,2255,"Plaquemines Port District, LA",23220752.0,15641368.0,3589521.0,19230889.0,42451641,NaN,1.009392e+09,965875627.0,43516535.0
3,5700,"Virginia, VA, Port of",3818058.0,30129060.0,1134530.0,31263590.0,35081648,NaN,NaN,NaN,NaN
4,2252,"Port of Greater Baton Rouge, LA",20193128.0,11506943.0,1500025.0,13006968.0,33200096,NaN,NaN,NaN,NaN


In [19]:
display(dryBulk.head(30))

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
0,DRY BULK,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,TOTAL,Short Tons,199,-27.3,153992
1,DRY BULK,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,FOREIGN,Short Tons,199,-47.7,87964
2,DRY BULK,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,IMPORTS,Short Tons,199,-47.7,87964
3,DRY BULK,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,EXPORTS,Short Tons,199,0.0,0
4,DRY BULK,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,DOMESTIC,Short Tons,199,51.4,66028
5,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,TOTAL,Short Tons,10,10.0,22520722
6,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,FOREIGN,Short Tons,10,8.9,19815315
7,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,IMPORTS,Short Tons,10,17.1,3943651
8,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,EXPORTS,Short Tons,10,7.1,15871664
9,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,DOMESTIC,Short Tons,10,18.6,2705407


In [15]:
dryBulk.shape

(240, 11)

In [20]:
dryBulk_hist.loc[dryBulk_hist['Port ID'] == 700]

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
175,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,DOMESTIC,Short Tons,10,18.6,2705407
176,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,EXPORTS,Short Tons,10,7.1,15871664
177,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,FOREIGN,Short Tons,10,8.9,19815315
178,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,IMPORTS,Short Tons,10,17.1,3943651
179,DRY BULK,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,TOTAL,Short Tons,10,10.0,22520722


### Dry Bulk Check

#### Internal Check

In [40]:
nbad=0
ngood=0
ntotal=0

for pid in dryBulk['Port ID'].unique():
    row = dryBulk[dryBulk['Port ID'] == pid]
    hld={}
    bad=False
    ntotal+=1
    for tt in row['Trade Type'].unique():
        hld[tt] = row[row['Trade Type'] == tt]['Volume'].values[0]
    frgn = hld['FOREIGN']
    dmst = hld['DOMESTIC']
    tot = hld['TOTAL']
    impt = hld['IMPORTS']
    expo = hld['EXPORTS']
    if frgn != impt + expo:
        print(f"Mismatch for Port ID {pid}: FOREIGN {frgn} vs IMPORTS+EXPORTS {impt+expo} IMPT {impt} EXPO {expo}")
        bad=True
    if tot != dmst + frgn:
        print(f"Mismatch for Port ID {pid}: TOTAL {tot} vs DOMESTIC+FOREIGN {dmst+frgn} DMST {dmst} FRGN {frgn}")
        bad=True

    if bad:
        nbad+=1
    else:
        ngood+=1

print(f"Checked {ntotal} ports with {nbad} mismatches and {ngood} good records.")

# Get value counts for the 'Trade Type' column
counts = dryBulk['Trade Type'].value_counts()

# Check if all counts are equal
all_equal = counts.nunique() == 1
print(f"All types have equal counts: {all_equal}")

# See which types have different counts
print(f"\nMin count: {counts.min()}")
print(f"Max count: {counts.max()}")
print(f"Difference: {counts.max() - counts.min()}")
#    print(f"Port ID {pid}, Total Volume from Trade Types: {tot}, Total Volume from TOTAL row: {row[row['Trade Type'] == 'TOTAL']['Volume'].values[0]}")

Checked 48 ports with 0 mismatches and 48 good records.
All types have equal counts: True

Min count: 48
Max count: 48
Difference: 0


In [37]:
dryBulk['Trade Type'].value_counts()

Trade Type
TOTAL       48
FOREIGN     48
IMPORTS     48
EXPORTS     48
DOMESTIC    48
Name: count, dtype: int64

#### Historical Check

In [35]:
dryBulk_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == "DRY BULK"].copy()
dryBulk_hist.shape
#dryBulk_hist['Port ID']= dryBulk_hist['Port ID'].astype(str).str.replace(",","")
dryBulk_hist['Volume']= dryBulk_hist['Volume'].astype(str).str.replace(",","").str.replace("nan","0").astype(int)

dryBulk_hist['Port Ranking']= dryBulk_hist['Port Ranking'].astype(int)

nchecks=0
nbad=0
for idx,row in dryBulk.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'] 
    nchecks+=1
    hst= dryBulk_hist.loc[(dryBulk_hist['Port ID']==prt) & (dryBulk_hist['Trade Type']==tp)]      
    for ii in row.index:
        if str(row[ii]).strip() != str(hst[ii].values[0]).strip() and str(ii) != "Percent Change":
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}")
             nbad+=1

dryBulk_Stns = set(dryBulk['Port_Name'].unique().tolist())
dryBulk_hist_Stns = set(dryBulk['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",dryBulk.shape)
print("Update Shape",dryBulk_hist.shape)
print("Dry Bulk Ports NEW not in HIST:", dryBulk_Stns - dryBulk_hist_Stns)
print("Dry Bulk Ports HIST not in NEW:", dryBulk_hist_Stns - dryBulk_Stns)

Completed 240 checks with 0 mismatches.
Update Shape (240, 11)
Update Shape (240, 11)
Dry Bulk Ports NEW not in HIST: set()
Dry Bulk Ports HIST not in NEW: set()


## Vessel Calls

In [68]:
df = xls.parse("Ports by ICST")

all_records = []

for idx,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
 #   tmp = df.loc[(df['PORT'] == port_id) ]
 #   print("Processing Port ID:", port_id, row['Port_Name'],tmp.shape)
    if df.loc[(df['PORT'] == port_id)].shape[0]==0:
        print("No data for port ID:",port_id,row['Port_Name'])
        continue
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'VESSEL CALLS')]
    
    nmiss=0
    for tt in ["Container","Other Freight Barge","Dry Bulk","Dry Bulk Barge","Other Freight"]:

        tmp = df.loc[(df['PORT'] == port_id) & (df['CATEGORY'].str.strip() == tt)]
        vol_minus_1 = hst[hst['Trade Type'] == tt]['Volume'].values[0] if not hst.empty else None              
       
        
        if tmp.shape[0] > 0:
            vol_in = tmp['INBOUND'].values[0]
            vol_out = tmp['OUTBOUND'].values[0]
            vol_mean = (vol_in + vol_out) / 2
            
        else:
            print("MIS ",port_id,tt,tmp.shape)
            vol_mean=0
            nmiss+=1

        if vol_minus_1 and vol_minus_1 > 0:
            val = tmp.loc[tmp['CATEGORY'].str.strip() == tt, 'INBOUND'].values[0] if not tmp.empty else 0   
            percent_change = ((vol_mean - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")

        hld = {
            'Cargo Type': 'VESSEL CALLS', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Vessel Calls', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': vol_mean
            }
        all_records.append(hld)
    if nmiss == 5:
        print("No data for Port ID:", port_id, row['Port_Name'])
vesselCalls = pd.DataFrame(all_records)
print("Vessel Calls shape:",vesselCalls.shape)

MIS  2393 Container (0, 6)
No historical volume for port ID Beaumont, TX 2393, trade type Container. Cannot calculate percent change.
No data for port ID: 2338 Cincinnati-Northern KY, Ports of
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Container. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Other Freight Barge. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Dry Bulk. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Dry Bulk Barge. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Other Freight. Cannot calculate percent change.
MIS  2436 Container (0, 6)
No historical volume for port ID Corpus Christi, TX 2436, trade type Container. Cannot calculate percent change.
MIS  3924 Container (0, 6)
No historica

In [72]:
hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == 3217) & (df_hist_year_minus_1['Cargo Type'] == 'VESSEL CALLS')]
display(hst)

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume


In [73]:
df_hist.loc[(df_hist['Port ID'] == 3217) & (df_hist['Cargo Type'] == 'VESSEL CALLS')]

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
495,VESSEL CALLS,3217,"Cleveland-Cuyahoga Port, OH",Great Lakes,2021,Ohio,Container,Vessel Calls,NaN,0.0,4
496,VESSEL CALLS,3217,"Cleveland-Cuyahoga Port, OH",Great Lakes,2021,Ohio,Dry Bulk,Vessel Calls,NaN,19.4,431.5
497,VESSEL CALLS,3217,"Cleveland-Cuyahoga Port, OH",Great Lakes,2021,Ohio,Dry Bulk Barge,Vessel Calls,NaN,38.8,193
498,VESSEL CALLS,3217,"Cleveland-Cuyahoga Port, OH",Great Lakes,2021,Ohio,Other Freight,Vessel Calls,NaN,14.3,36
499,VESSEL CALLS,3217,"Cleveland-Cuyahoga Port, OH",Great Lakes,2021,Ohio,Other Freight Barge,Vessel Calls,NaN,5.7,18.5


### Vessel Calls Check

In [69]:
vc_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == "VESSEL CALLS"].copy()

print(vc_hist.shape)
vc_hist['Port ID']= vc_hist['Port ID'].astype(str).str.replace(",","").astype(int)
vc_hist['Volume']= vc_hist['Volume'].astype(str).str.replace(",","").str.replace("nan","0").astype(float)

B=vesselCalls.copy()
A=vc_hist.copy()
nchecks=0
nbad=0
for idx,row in A.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'].strip()
    nchecks+=1
    hst= B.loc[(B['Port ID']==prt) & (B['Trade Type'].str.strip()==tp)] 
    if hst.shape[0] > 1:
        print("Multiple HIST records for Port ID:",prt," Trade Type:",tp,row)
    elif hst.shape[0] == 0:
        print("No mmm NEW data for Port ID:", prt, row['Port_Name'], " Trade Type:", tp)
        continue

    if len(row)==0:
        print("No NEW data for Port ID:", prt, " Trade Type:", tp)
        continue
    for ii in row.index:
        if str(row[ii]).strip() != str(hst[ii].values[0]).strip() and str(ii) != "Port Ranking":
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: Old={row[ii]} vs New={hst[ii].values[0]}")
             nbad+=1

vesselCalls_Stns = set(vesselCalls['Port_Name'].unique().tolist())
vc_hist_Stns = set(vc_hist['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",vesselCalls.shape)
print("  Hist Shape",vc_hist.shape)
print("Vessel Calls Ports NEW not in HIST:", vesselCalls_Stns - vc_hist_Stns)
print("Vessel Calls Ports HIST not in NEW:", vc_hist_Stns - vesselCalls_Stns)

(240, 11)
Mismatch for Port ID 3217, Trade Type Dry Bulk, Field Percent Change: Old=19.4 vs New=0.0
Mismatch for Port ID 3217, Trade Type Dry Bulk Barge, Field Percent Change: Old=38.8 vs New=0.0
Mismatch for Port ID 3217, Trade Type Other Freight, Field Percent Change: Old=14.3 vs New=0.0
Mismatch for Port ID 3217, Trade Type Other Freight Barge, Field Percent Change: Old=5.7 vs New=0.0
Mismatch for Port ID 1983, Trade Type Container, Field Percent Change: Old=-2.2 vs New=0.0
Mismatch for Port ID 1983, Trade Type Dry Bulk, Field Percent Change: Old=450.0 vs New=0.0
Mismatch for Port ID 1983, Trade Type Dry Bulk Barge, Field Percent Change: Old=-21.3 vs New=0.0
Mismatch for Port ID 1983, Trade Type Other Freight, Field Percent Change: Old=-47.3 vs New=0.0
Mismatch for Port ID 1983, Trade Type Other Freight Barge, Field Percent Change: Old=-8.3 vs New=0.0
Mismatch for Port ID 1992, Trade Type Dry Bulk, Field Percent Change: Old=0.0 vs New=-100.0
Mismatch for Port ID 3204, Trade Type Dry

## Top 5 Commodities

In [ ]:
## Fix this to get commodities out of 2021 port performance list


df = xls.parse("Ports by Commodity")

all_records = []

for _,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
    hld=df.loc[df['PORT'] == port_id]
    total = hld['TOTAL'].sum()
    hld=hld.sort_values(by="TOTAL", ascending=False).reset_index(drop=True).iloc[:5]
    for idx,row2 in hld.iterrows():
        tt=row2['Commodity Name'].strip() 
        volume=row2['TOTAL']
        hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == tt)]
        vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
        if vol_minus_1 and vol_minus_1 > 0:
            val = row2['TOTAL']   
            percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")




        tmp = {
            'Cargo Type': 'TOP 5 COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': volume
            }
        if volume> 0:
           all_records.append(tmp)
## Add Total
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == 'TOTAL')]
    vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
    if vol_minus_1 and vol_minus_1 > 0:
        val = total   
        percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
        percent_change = round(percent_change, 1)
    else:
        percent_change = 0
        print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")

    tmp = {
            'Cargo Type': 'TOP 5 COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':'TOTAL', 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':None,
            'Volume': total
            }
    if total > 0:
       all_records.append(tmp)

top5Com = pd.DataFrame(all_records)

print("Top 5 Commodities shape:",top5Com.shape)


    
    

No historical volume for port ID Alaska, AK Port of 4820, trade type Distillate Fuel Oil. Cannot calculate percent change.
No historical volume for port ID Boston, MA 149, trade type Kerosene. Cannot calculate percent change.
No historical volume for port ID Boston, MA 149, trade type Alcohols. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Iron Ore. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Limestone. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Salt. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Cement & Concrete. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type I&S Plates & Sheets. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyaho

### Top 5 Commodity Check

In [36]:
top5_hist=df_hist_year.loc[df_hist_year['Cargo Type'] == 'TOP 5 COMMODITIES']
top5_hist.head()
#top5_hist["Port ID"] = top5_hist["Port ID"].str.replace(",","").astype(int)
top5_hist["Volume"] = top5_hist["Volume"].str.replace(",","").astype(int)
B=top5Com.copy()
A=top5_hist.copy()
nchecks=0
nbad=0
for idx,row in A.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'].strip()
    nchecks+=1
    hst= B.loc[(B['Port ID']==prt) & (B['Trade Type'].str.strip()==tp)] 
    if hst.shape[0] > 1:
        print("Multiple HIST records for Port ID:",prt," Trade Type:",tp,row)
    elif hst.shape[0] == 0:
        print("No mmm NEW data for Port ID:", prt, row['Port_Name'], " Trade Type:", tp)
        continue

    if len(row)==0:
        print("No NEW data for Port ID:", prt, " Trade Type:", tp)
        continue
    for ii in row.index:
        if is_number(row[ii]) and is_number(hst[ii].values[0]):
            if abs(float(row[ii]) - float(hst[ii].values[0])) > 0.0 and str(ii) != "Port Ranking":
                print(f"Numeric Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}  ")
                nbad+=1
                if ii == 'Volume':
                    badsn[(prt,tp)] = (row[ii], hst[ii].values[0])
        elif str(row[ii]).strip() != str(hst[ii].values[0]).strip()  and str(ii) != "Port Ranking":
             print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: Hist={row[ii]} vs Row={hst[ii].values[0]}")
             nbad+=1
        

top5Com_Stns = set(top5Com['Port_Name'].unique().tolist())
top5_hist_Stns = set(top5_hist['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",top5Com.shape)
print("  Hist Shape",top5_hist.shape)
print("Vessel Calls Ports NEW not in HIST:", top5Com_Stns - top5_hist_Stns)
print("Vessel Calls Ports HIST not in NEW:", top5_hist_Stns - top5Com_Stns)

Numeric Mismatch for Port ID 4820, Trade Type Distillate Fuel Oil, Field Percent Change: New=180.7 vs Hist=0.0  
Numeric Mismatch for Port ID 149, Trade Type Alcohols, Field Percent Change: New=26.8 vs Hist=0.0  
Numeric Mismatch for Port ID 149, Trade Type Kerosene, Field Percent Change: New=58.4 vs Hist=0.0  
Numeric Mismatch for Port ID 3217, Trade Type Cement & Concrete, Field Percent Change: New=1.4 vs Hist=0.0  
Numeric Mismatch for Port ID 3217, Trade Type I&S Plates & Sheets, Field Percent Change: New=45.4 vs Hist=0.0  
Numeric Mismatch for Port ID 3217, Trade Type Iron Ore, Field Percent Change: New=44.0 vs Hist=0.0  
Numeric Mismatch for Port ID 3217, Trade Type Limestone, Field Percent Change: New=18.6 vs Hist=0.0  
Numeric Mismatch for Port ID 3217, Trade Type Salt, Field Percent Change: New=12.1 vs Hist=0.0  
Numeric Mismatch for Port ID 2436, Trade Type Naphtha & Solvents, Field Percent Change: New=56.5 vs Hist=0.0  
Numeric Mismatch for Port ID 3924, Trade Type Petroleum

/tmp/ipykernel_169299/3468564628.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top5_hist["Volume"] = top5_hist["Volume"].str.replace(",","").astype(int)


## Top 5 Farm/Agriculture Commodities

In [34]:
top5Ag_hist = df_hist.loc[df_hist['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES']
ag_commodities_hist = top5Ag_hist['Trade Type'].unique().tolist()
df = xls.parse("Ports by Commodity")
all_records = []

for _,row in ports_hist.iterrows():
    port_id = int(row['Port ID'])
 #   hld=top5Ag.loc[top5Ag['PORT'] == port_id]
    total = df.loc[(df['PORT'] == port_id) & (df['Commodity Group'] >= 6000) & (df['Commodity Group'] < 7000),'TOTAL'].sum()  
    hld = df.loc[(df['PORT'] == port_id) & (df['Commodity Group'] >= 6000) & (df['Commodity Group'] < 7000)]
    hld=hld.sort_values(by="TOTAL", ascending=False).reset_index(drop=True).iloc[:5]
    for idx,row2 in hld.iterrows():
        tt=row2['Commodity Name'].strip() 
        volume=row2['TOTAL']
        hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == tt)]
        vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
        if vol_minus_1 and vol_minus_1 > 0:
            val = row2['TOTAL']   
            percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
            percent_change = round(percent_change, 1)
        else:
            percent_change = 0
            print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")


        tmp = {
            'Cargo Type': 'TOP 5 FOOD/FARM COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':tt, 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': volume
            }
        if volume> 0:
           all_records.append(tmp)
## Add Total
    hst = df_hist_year_minus_1.loc[(df_hist_year_minus_1['Port ID'] == port_id) & (df_hist_year_minus_1['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES') & (df_hist_year_minus_1['Trade Type'] == 'TOTAL')]
    vol_minus_1 = hst['Volume'].values[0] if not hst.empty else 0
    if vol_minus_1 and vol_minus_1 > 0:
        val = total   
        percent_change = ((val - vol_minus_1) / vol_minus_1) * 100
        percent_change = round(percent_change, 1)
    else:
        percent_change = 0
        print(f"No historical volume for port ID {row['Port_Name']} {port_id}, trade type {tt}. Cannot calculate percent change.")



    tmp = {
            'Cargo Type': 'TOP 5 FOOD/FARM COMMODITIES', 
            'Port ID': port_id, 
            'Port_Name': row['Port_Name'], 
            'Region': row['Region'],
            'Reporting Year': REPORTING_YEAR,
            'State': row['State'], 
            'Trade Type':'TOTAL', 
            'Units':'Short Tons', 
            'Port Ranking': None,
            'Percent Change':percent_change,
            'Volume': total
            }
    if total > 0:
       all_records.append(tmp)

top5Ag = pd.DataFrame(all_records)

print("Top 5 Farm/Food Commodities shape:",top5Ag.shape)



No historical volume for port ID Alaska, AK Port of 4820, trade type Fish (Not Shellfish). Cannot calculate percent change.
No historical volume for port ID Baltimore, MD 700, trade type Alcoholic Beverages. Cannot calculate percent change.
No historical volume for port ID Beaumont, TX 2393, trade type Vegetable Oils. Cannot calculate percent change.
No historical volume for port ID Beaumont, TX 2393, trade type Meat, Fresh, Frozen. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Vegetables & Prod.. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Food Products NEC. Cannot calculate percent change.
No historical volume for port ID Cleveland-Cuyahoga Port, OH 3217, trade type Food Products NEC. Cannot calculate percent change.
No historical volume for port ID Corpus Christi, TX 2436, trade type Fish (Not Shellfish). Cannot calculate percent change.
No historical v

### Top 5 Farm/Agriculture Check

In [35]:
from pandas.api.types import is_number
top5Ag_hist = df_hist_year.loc[df_hist_year['Cargo Type'] == 'TOP 5 FOOD/FARM COMMODITIES']
#ag_commodities_hist = top5Ag_hist['Trade Type'].unique().tolist()


#top5Ag_hist["Port ID"] = top5Ag_hist["Port ID"].str.replace(",","").astype(int)
top5Ag_hist["Volume"] = top5Ag_hist["Volume"].str.replace(",","").astype(int)
B=top5Ag.copy()
A=top5Ag_hist.copy()
nchecks=0
nbad=0
badsn={}
for idx,row in A.iterrows():
    prt=row['Port ID']
    tp=row['Trade Type'].strip()
    nchecks+=1
    hst= B.loc[(B['Port ID']==prt) & (B['Trade Type'].str.strip()==tp)] 
    if hst.shape[0] > 1:
        print("Multiple HIST records for Port ID:",prt," Trade Type:",tp,row)
    elif hst.shape[0] == 0:
        print("No mmm NEW data for Port ID:", prt, row['Port_Name'], " Trade Type:", tp)
        continue

    if len(row)==0:
        print("No NEW data for Port ID:", prt, " Trade Type:", tp)
        continue
    for ii in row.index:
        
        if is_number(row[ii]) and is_number(hst[ii].values[0]):
            if abs(float(row[ii]) - float(hst[ii].values[0])) > 0.0 and str(ii) != "Port Ranking":
                print(f"Numeric Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}  ")
                nbad+=1
                if ii == 'Volume':
                    badsn[(prt,tp)] = (row[ii], hst[ii].values[0])
        elif str(row[ii]).strip() != str(hst[ii].values[0]).strip() and str(ii) != "Port Ranking":
       #      print(f"Mismatch for Port ID {prt}, Trade Type {tp}, Field {ii}: New={row[ii]} vs Hist={hst[ii].values[0]}  diff (H-U) {row[ii] - hst[ii].values[0]}")
             nbad+=1
             if ii == 'Volume':
                 badsn[(prt,tp)] = (row[ii], hst[ii].values[0])
        

top5Ag_Stns = set(top5Ag['Port_Name'].unique().tolist())
top5Ag_hist_Stns = set(top5Ag_hist['Port_Name'].unique().tolist())



print(f"Completed {nchecks} checks with {nbad} mismatches.")
print("Update Shape",top5Ag.shape)
print("  Hist Shape",top5Ag_hist.shape)
print("Vessel Calls Ports NEW not in HIST:", top5Ag_Stns - top5Ag_hist_Stns)
print("Vessel Calls Ports HIST not in NEW:", top5Ag_hist_Stns - top5Ag_Stns)


Numeric Mismatch for Port ID 4820, Trade Type Fish (Not Shellfish), Field Percent Change: New=878.0 vs Hist=0.0  
Numeric Mismatch for Port ID 4820, Trade Type TOTAL, Field Percent Change: New=11.4 vs Hist=17.1  
Numeric Mismatch for Port ID 700, Trade Type Alcoholic Beverages, Field Percent Change: New=-1.9 vs Hist=0.0  
Numeric Mismatch for Port ID 700, Trade Type TOTAL, Field Percent Change: New=59.3 vs Hist=-4.5  
Numeric Mismatch for Port ID 2393, Trade Type Meat, Fresh, Frozen, Field Percent Change: New=619.0 vs Hist=0.0  
Numeric Mismatch for Port ID 2393, Trade Type TOTAL, Field Percent Change: New=184.3 vs Hist=67.2  
Numeric Mismatch for Port ID 2393, Trade Type Vegetable Oils, Field Percent Change: New=184.3 vs Hist=0.0  
Numeric Mismatch for Port ID 149, Trade Type TOTAL, Field Percent Change: New=18.9 vs Hist=-7.7  
Numeric Mismatch for Port ID 3217, Trade Type TOTAL, Field Percent Change: New=-78.8 vs Hist=0.0  
Numeric Mismatch for Port ID 3217, Trade Type Vegetables & P

/tmp/ipykernel_169299/1076671813.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top5Ag_hist["Volume"] = top5Ag_hist["Volume"].str.replace(",","").astype(int)


In [26]:
print(top5Ag.dtypes)
print(top5Ag_hist.dtypes)

Cargo Type         object
Port ID             int64
Port_Name          object
Region             object
Reporting Year      int64
State              object
Trade Type         object
Units              object
Port Ranking       object
Percent Change    float64
Volume              int64
dtype: object
Cargo Type         object
Port ID             int64
Port_Name          object
Region             object
Reporting Year      int64
State              object
Trade Type         object
Units              object
Port Ranking      float64
Percent Change    float64
Volume              int64
dtype: object


In [18]:
hn={}
for k,v in badsn.items():
 #   print(f"Port ID {k[0]}, Trade Type {k[1]}: New={v[0]} vs Hist={v[1]}  diff (H-U) {v[0]-v[1]}")
    hn[k[0]] = v

ho={}
for k,v in badsn.items():
 #   print(f"Port ID {k[0]}, Trade Type {k[1]}: New={v[0]} vs Hist={v[1]}  diff (H-U) {v[0]-v[1]}")
    ho[k[0]] = v

for p,v in ho.items():
    if p in hn:
         print(f"Port ID {p}: OLD={v[0]} Org={v[1]} ")
         print(f"Port ID {p}: New={hn[p][0]} Org={hn[p][1]}  Dif=Org={hn[p][0] - hn[p][1]}\n")
    

## Container Activity

In [24]:

# ======================================================
# 3. CONTAINER ACTIVITY (TEUs)
# ======================================================
df =pd.read_csv("PortDataExport.csv")
df.columns = df.columns.astype(str).str.strip().str.upper()
print("CONTAINER ACTIVITY COLUMNS:", df.columns.tolist())
# for _, r in df.iterrows():
#     port = r["PORT NAME"]
#     rank = None

#     flows = {
#         "TOTAL": r.get("TOTAL"),
#         "IMPORTS": r.get("IMPORTS"),
#         "EXPORTS": r.get("EXPORTS"),
#         "EMPTY": r.get("EMPTY")
#     }

    # for trade_type, value in flows.items():
    #     if pd.notna(value):
    #         records.append({
    #             "Cargo Type": "CONTAINER ACTIVITY",
    #             "Port_Name": port,
    #             "Reporting Year": REPORTING_YEAR,
    #             "Trade Type": trade_type,
    #             "Units": "TEUs",
    #             "Volume": value,
    #             "Port Ranking": None
    #         })


CONTAINER ACTIVITY COLUMNS: ['CARGO TYPE', 'PORT ID', 'PORT_NAME', 'REGION', 'REPORTING YEAR', 'STATE', 'TRADE TYPE', 'UNITS', 'PORT RANKING', 'PERCENT CHANGE', 'VOLUME']


In [25]:
df.columns

Index(['CARGO TYPE', 'PORT ID', 'PORT_NAME', 'REGION', 'REPORTING YEAR',
       'STATE', 'TRADE TYPE', 'UNITS', 'PORT RANKING', 'PERCENT CHANGE',
       'VOLUME'],
      dtype='object')

In [21]:
a=df.loc[df['CARGO TYPE'] == "CONTAINER"] 

In [22]:
a.head(50)

,CARGO TYPE,PORT ID,PORT_NAME,REGION,REPORTING YEAR,STATE,TRADE TYPE,UNITS,PORT RANKING,PERCENT CHANGE,VOLUME
1120,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2018,New Jersey and New York,EMPTY,Container TEUs,3.0,-65.723763779,"4,959.6"
1121,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2018,New Jersey and New York,EXPORTS,Container TEUs,3.0,6.154284058,"1,500,172.82"
1122,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2018,New Jersey and New York,IMPORTS,Container TEUs,3.0,9.944265192,"3,782,318.63"
1123,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2018,New Jersey and New York,TOTAL,Container TEUs,3.0,8.840711710,"5,282,491.45"
1124,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2019,New Jersey and New York,EMPTY,Container TEUs,2.0,134.803209936,"11,645.3"
1125,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2019,New Jersey and New York,EXPORTS,Container TEUs,2.0,-2.400755401,"1,464,157.34"
1126,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2019,New Jersey and New York,IMPORTS,Container TEUs,2.0,-0.127410207,"3,777,499.57"
1127,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2019,New Jersey and New York,TOTAL,Container TEUs,2.0,-0.773016679,"5,241,656.91"
1128,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2020,New Jersey and New York,EMPTY,Container TEUs,3.0,-100,0
1129,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2020,New Jersey and New York,EXPORTS,Container TEUs,3.0,-14.395832623,"1,253,379.7"


In [23]:
a['REPORTING YEAR'].value_counts()

REPORTING YEAR
2022    156
2021    152
2020    112
2018    104
2019    104
Name: count, dtype: int64

In [24]:
print(df.loc[df['CARGO TYPE'] == "TOTAL TONNAGE",'PORT_NAME'].value_counts())
print( df.loc[df['CARGO TYPE'] == "DRY BULK",'PORT_NAME'].value_counts())


PORT_NAME
New York, NY & NJ                         25
Philadelphia Regional Port, PA            25
Wilmington, DE                            25
Wilmington, NC                            25
Baltimore, MD                             25
Port of Charleston, SC, SC                25
Savannah, GA Port of                      25
Jacksonville, FL                          25
Port Everglades, FL                       25
San Juan, PR                              25
PortMiami, FL                             25
Houston Port Authority, TX                25
Tampa Port Authority, FL                  25
Mobile, AL                                25
Lake Charles Harbor District, LA          25
Tacoma, WA                                25
New Orleans, LA                           25
Greater Baton Rouge, LA Port of           25
South Louisiana, LA, Port of              25
Plaquemines Port District, LA             25
Pittsburgh, PA Port of                    25
Port Freeport, TX                         25


#### 2023 Containers

In [4]:
df = pd.read_excel("2023 Annual TEUs from US Army.xlsx", header=[0, 1, 2])

In [32]:
df.head()

PORT CODE                                          PORT NAME  \
  Unnamed: 0_level_1                                 Unnamed: 1_level_1   
  Unnamed: 0_level_2                                 Unnamed: 1_level_2   
0               4120                            Port of Los Angeles, CA   
1                398  Port Authority of New York and New Jersey, NY ...   
2               4110                             Port of Long Beach, CA   
3                776                               Port of Savannah, GA   
4               2031     Port of Houston Authority of Harris County, TX   

               STATE DOMESTIC                                            \
  Unnamed: 2_level_1 InBound         OutBound            Total Domestic   
  Unnamed: 2_level_2   Loaded  Empty    Loaded Empty Unnamed: 7_level_2   
0                 CA        0      0         0     0                  0   
1              NY,NJ    26987   5449     26991  5452              64878   
2                 CA    37627  35206    265626  1113             339571   
3                 GA        0      0         0     0                  0   
4                 TX      594      0         0    72                666   

   FOREIGN                            Grand  
  InBound  OutBound  Total Foreign    Total  
    Loaded    Loaded        Loaded   Loaded  
0  4456523   1270261       5726784  5726784  
1  4035650   1307534       5343184  5397161  
2  3788876    997507       4786383  5089636  
3  2392604   1382170       3774774  3774774  
4  1804119   1405185       3209303  3209897

In [6]:
a=df.columns.tolist()
colsNew = []
for b in a:
    final=""

    for c in b:
        if "Unnamed:" in c:
            continue
        else:
            final += c.strip() + " "
    colsNew.append(final.strip())
    
df.columns = colsNew


In [7]:
df.columns

Index(['PORT CODE', 'PORT NAME', 'STATE', 'DOMESTIC InBound Loaded',
       'DOMESTIC InBound Empty', 'DOMESTIC OutBound Loaded',
       'DOMESTIC OutBound Empty', 'DOMESTIC Total Domestic',
       'FOREIGN InBound Loaded', 'FOREIGN OutBound Loaded',
       'FOREIGN Total Foreign Loaded', 'Grand Total Loaded'],
      dtype='object')

In [8]:
df['Imports'] = df['DOMESTIC InBound Loaded']+ df['FOREIGN InBound Loaded'] 
df['Exports'] = df['DOMESTIC OutBound Loaded']+ df['FOREIGN OutBound Loaded'] 
df['Empty'] = df['DOMESTIC InBound Empty']

In [9]:
df_contain = df[['PORT CODE', 'PORT NAME', 'STATE','Grand Total Loaded', 'Imports','Exports', 'Empty']].copy()
df_contain.head()

,PORT CODE,PORT NAME,STATE,Grand Total Loaded,Imports,Exports,Empty
0,4120,"Port of Los Angeles, CA",CA,5726784,4456523,1270261,0
1,398,"Port Authority of New York and New Jersey, NY ...","NY,NJ",5397161,4062637,1334525,5449
2,4110,"Port of Long Beach, CA",CA,5089636,3826503,1263133,35206
3,776,"Port of Savannah, GA",GA,3774774,2392604,1382170,0
4,2031,"Port of Houston Authority of Harris County, TX",TX,3209897,1804713,1405185,0


In [17]:
df_contain['PORT CODE'] = df_contain['PORT CODE'].astype(str).str.strip()   
for _,rows in ports_hist.iterrows():
    pid = rows['Port ID']
    hld = df_contain.loc[(df_contain['PORT CODE'] == pid) | (df_contain['PORT NAME'].str.strip() == rows['Port_Name'].strip())  ]
    if hld.shape[0] == 0:
        print("No data for Port ID:", pid, rows['Port_Name'])

No data for Port ID: 149 Boston, MA
No data for Port ID: 2338 Cincinnati-Northern KY, Ports of
No data for Port ID: 2083 Gulfport, MS
No data for Port ID: 2348 Huntington-Tristate, KY, OH, WV
No data for Port ID: 4626 Kalama, WA Port of
No data for Port ID: 2306 Mid-America Port Commission
No data for Port ID: 2351 New Bourbon Port, MO
No data for Port ID: 3743 Northern Indiana, IN
No data for Port ID: 2358 Pittsburgh, PA Port of
No data for Port ID: 2255 Plaquemines Port District, LA
No data for Port ID: 2363 Southern Indiana Maritime District, IN
No data for Port ID: 2367 St. Louis, MO and IL
No data for Port ID: 3204 Toledo-Lucas County Port, OH
No data for Port ID: 3926 Two Harbors, MN


In [27]:
for _,row in df_contain.iterrows():
    print(row['PORT CODE'],":",row['PORT NAME'])

4120 : Port of Los Angeles, CA
398 : Port Authority of New York and New Jersey, NY & NJ
4110 : Port of Long Beach, CA
776 : Port of Savannah, GA
2031 : Port of Houston Authority of Harris County, TX
5700 : Port of Virginia, VA
775 : Port of Charleston, SC
4344 : Port of Oakland, CA
4719 : Tacoma, WA
2017 : Jacksonville, FL
1992 : PortMiami, FL
1913 : San Juan, PR
4722 : Port of Seattle, WA
700 : Baltimore, MD
4421 : Honolulu, O'ahu, HI
552 : Philadelphia Regional Port Authority, PA
1911 : Port Everglades, FL
2032 : Mobile, AL
4820 : Port of Alaska, AK
2251 : Port of New Orleans, LA
766 : Wilmington, NC
90 : Port of Boston, MA
555 : Wilmington, DE
4151 : Oxnard Harbor District, CA
550 : South Jersey Port Corporation, NJ
2021 : Tampa Port Authority, FL
2440 : Port of Gulfport, MS
1983 : Port of Palm Beach District, FL
4823 : Juneau, AK
2437 : Manatee County Port Authority, FL
1912 : Guaynabo, PR
4827 : Ketchikan, AK
4646 : Port of Portland, OR
4410 : Kahului, Maui, HI
4826 : Petersburg, 

In [26]:
for _,row in ports_hist.iterrows():
    print(row['Port ID'],":", row['Port_Name'])

4820 : Alaska, AK Port of
700 : Baltimore, MD
2393 : Beaumont, TX
149 : Boston, MA
2338 : Cincinnati-Northern KY, Ports of
3217 : Cleveland-Cuyahoga Port, OH
2436 : Corpus Christi, TX
3924 : Duluth-Superior, MN and WI
4719 : Tacoma, WA
2252 : Greater Baton Rouge, LA Port of
2083 : Gulfport, MS
4421 : Honolulu, O'ahu, HI
2031 : Houston Port Authority, TX
2348 : Huntington-Tristate, KY, OH, WV
2017 : Jacksonville, FL
4626 : Kalama, WA Port of
2248 : Lake Charles Harbor District, LA
4110 : Long Beach, CA Port of
4622 : Longview, WA Port of
4120 : Los Angeles, CA Port of
2306 : Mid-America Port Commission
2032 : Mobile, AL
2351 : New Bourbon Port, MO
2251 : New Orleans, LA
398 : New York, NY & NJ
3743 : Northern Indiana, IN
4344 : Oakland, CA Port of
552 : Philadelphia Regional Port, PA
2358 : Pittsburgh, PA Port of
2255 : Plaquemines Port District, LA
2416 : Port Arthur, TX
1911 : Port Everglades, FL
2408 : Port Freeport, TX
775 : Port of Charleston, SC
1983 : Port of Palm Beach District,

In [36]:
dfA = pd.read_excel("2023 Annual TEUs from US Army.xlsx", header=[0, 1, 2],sheet_name=['2023 TEUs by Port','2023 TEUs by Name','2023 TEUs by State'])
dfP = dfA['2023 TEUs by Port']
dfN = dfA['2023 TEUs by Name']
dfS = dfA['2023 TEUs by State']



In [52]:
df22 = pd.read_excel("WCSC 2022 Container traffic.xlsx", header=[0, 1, 2],sheet_name=['2022 TEUs by Port'])


In [54]:
df22 = df22['2022 TEUs by Port']

In [59]:
df22=fixCOls(df22)

In [72]:
a=df22['PORT NAME'].unique().tolist()

In [64]:
b=ports_hist['Port_Name'].unique().tolist()

In [73]:
for i in sorted(a):
    print(i)

Baltimore, MD
Beaumont, TX
Bellingham, WA
Bethel, AK
Brevig Mission, AK
Buffalo, NY
Calhoun Port Authority, TX
Canaveral Port District, FL
Cheboygan, MI
Clallam County Port District, WA
Cleveland-Cuyahoga County, OH
Cordova, AK
Corpus Christi, TX
DeTour, MI
Detroit/Wayne County Port Authority, MI
Dillingham, AK
Duluth-Superior, MN and WI
Fall River Port Authority, MA
False Pass, AK
Galveston, TX
Guayanilla, PR
Guaynabo, PR
Haines Borough, AK
Hilo, Hawai'i, HI
Honolulu, O'ahu, HI
Hoonah, AK
Hooper Bay, AK
Illinois International Port District, IL
Jacksonville, FL
Jefferson County Port District, WA
Juneau, AK
Kahului, Maui, HI
Kake, AK
Kaumalapau, Lana'i, HI
Kaunakakai, Moloka'i, HI
Kawaihae, Hawai'i, HI
Ketchikan, AK
Kivalina, AK
Kodiak, AK
Lake Charles Harbor and Terminal District, LA
Manatee County Port Authority, FL
Metlakatla, AK
Mobile, AL
Morehead City, NC
Nawiliwili, Kaua'i, HI
New Bedford Port Authority, MA
Nome, AK
Ocean Highway and Port Authority, FL
Old Harbor, AK
Oxnard Harbo

In [67]:
print(b)

['Alaska, AK Port of', 'Baltimore, MD', 'Beaumont, TX', 'Boston, MA', 'Cincinnati-Northern KY, Ports of', 'Cleveland-Cuyahoga Port, OH', 'Corpus Christi, TX', 'Duluth-Superior, MN and WI', 'Tacoma, WA', 'Greater Baton Rouge, LA Port of', 'Gulfport, MS', "Honolulu, O'ahu, HI", 'Houston Port Authority, TX', 'Huntington-Tristate, KY, OH, WV', 'Jacksonville, FL', 'Kalama, WA Port of', 'Lake Charles Harbor District, LA', 'Long Beach, CA Port of', 'Longview, WA Port of', 'Los Angeles, CA Port of', 'Mid-America Port Commission', 'Mobile, AL', 'New Bourbon Port, MO', 'New Orleans, LA', 'New York, NY & NJ', 'Northern Indiana, IN', 'Oakland, CA Port of', 'Philadelphia Regional Port, PA', 'Pittsburgh, PA Port of', 'Plaquemines Port District, LA', 'Port Arthur, TX', 'Port Everglades, FL', 'Port Freeport, TX', 'Port of Charleston, SC', 'Port of Palm Beach District, FL', 'Port of Portland, OR', 'PortMiami, FL', 'San Juan, PR', 'Savannah, GA Port of', 'Seattle, WA', 'South Jersey Port, NJ', 'South Lo

Cincinnati-Northern KY, Ports of
Huntington-Tristate, KY, OH, WV
Kalama, WA Port of
Mid-America Port Commission
New Bourbon Port, MO
Northern Indiana, IN
Pittsburgh, PA Port of
Southern Indiana Maritime District, IN
St. Louis, MO and IL
Toledo-Lucas County Port, OH
Two Harbors, MN

2338	Cincinnati-Northern KY, Ports of
2348	Huntington-Tristate, KY, OH, WV
4626	Kalama, WA Port of
2306	Mid-America Port Commission
2351	New Bourbon Port, MO
3743	Northern Indiana, IN
2358	Pittsburgh, PA Port of
2363	Southern Indiana Maritime District, IN
2367	St. Louis, MO and IL
3204	Toledo-Lucas County Port, OH
3926	Two Harbors, MN

2255	Plaquemines Port District, LA

In [71]:
df_hist['Port_Name'].nunique()

54

In [32]:
dfP['2023 TEUs by Port'].columns

MultiIndex([('PORT CODE', 'Unnamed: 0_level_1', 'Unnamed: 0_level_2'),
            ('PORT NAME', 'Unnamed: 1_level_1', 'Unnamed: 1_level_2'),
            (    'STATE', 'Unnamed: 2_level_1', 'Unnamed: 2_level_2'),
            ( 'DOMESTIC',           'InBound ',             'Loaded'),
            ( 'DOMESTIC',           'InBound ',              'Empty'),
            ( 'DOMESTIC',          'OutBound ',             'Loaded'),
            ( 'DOMESTIC',          'OutBound ',              'Empty'),
            ( 'DOMESTIC',     'Total Domestic', 'Unnamed: 7_level_2'),
            (  'FOREIGN',           'InBound ',             'Loaded'),
            (  'FOREIGN',          'OutBound ',             'Loaded'),
            (  'FOREIGN',      'Total Foreign',             'Loaded'),
            (    'Grand',              'Total',             'Loaded')],
           )

In [39]:

def fixCOls(df):
    a=df.columns.tolist()
    colsNew = []
    for b in a:
        final=""
        for c in b:
            if "Unnamed:" in c:
                continue
            else:
                final += c.strip() + " "
        colsNew.append(final.strip())
        
    df.columns = colsNew
    return df
dfP = fixCOls(dfP)
dfN = fixCOls(dfN)
dfS = fixCOls(dfS)

In [48]:
sorted(df['PORT NAME'].unique().tolist())

['Albany Port District, NY',
 'Baltimore, MD',
 'Beaumont, TX',
 'Bellingham, WA',
 'Bethel, AK',
 'Brevig Mission, AK',
 'Buffalo, NY',
 'Calhoun Port Authority, TX',
 'Canaveral Port District, FL',
 'Cheboygan, MI',
 'Clallam County Port District, WA',
 'Clatsop County Port District, OR',
 'Cleveland-Cuyahoga County, OH',
 'Cordova, AK',
 'Corpus Christi, TX',
 'Dillingham, AK',
 'Duluth-Superior, MN and WI',
 'Fall River Port Authority, MA',
 'False Pass, AK',
 'Galveston, TX',
 'Guayanilla, PR',
 'Guaynabo, PR',
 'Haines Borough, AK',
 "Hilo, Hawai'i, HI",
 "Honolulu, O'ahu, HI",
 'Hoonah, AK',
 'Illinois International Port District, IL',
 'Jacksonville, FL',
 'Juneau, AK',
 'Kahului, Maui, HI',
 'Kake, AK',
 "Kaumalapau, Lana'i, HI",
 "Kaunakakai, Moloka'i, HI",
 "Kawaihae, Hawai'i, HI",
 'Ketchikan, AK',
 'Kivalina, AK',
 'Kodiak, AK',
 'Lake Charles Harbor and Terminal District, LA',
 'Manatee County Port Authority, FL',
 'Metlakatla, AK',
 'Mobile, AL',
 'Morehead City, NC',
 "

In [49]:
sorted(ports_hist['Port_Name'].unique().tolist())

['Alaska, AK Port of',
 'Baltimore, MD',
 'Beaumont, TX',
 'Boston, MA',
 'Cincinnati-Northern KY, Ports of',
 'Cleveland-Cuyahoga Port, OH',
 'Corpus Christi, TX',
 'Duluth-Superior, MN and WI',
 'Greater Baton Rouge, LA Port of',
 'Gulfport, MS',
 "Honolulu, O'ahu, HI",
 'Houston Port Authority, TX',
 'Huntington-Tristate, KY, OH, WV',
 'Jacksonville, FL',
 'Kalama, WA Port of',
 'Lake Charles Harbor District, LA',
 'Long Beach, CA Port of',
 'Longview, WA Port of',
 'Los Angeles, CA Port of',
 'Mid-America Port Commission',
 'Mobile, AL',
 'New Bourbon Port, MO',
 'New Orleans, LA',
 'New York, NY & NJ',
 'Northern Indiana, IN',
 'Oakland, CA Port of',
 'Philadelphia Regional Port, PA',
 'Pittsburgh, PA Port of',
 'Plaquemines Port District, LA',
 'Port Arthur, TX',
 'Port Everglades, FL',
 'Port Freeport, TX',
 'Port of Charleston, SC',
 'Port of Palm Beach District, FL',
 'Port of Portland, OR',
 'PortMiami, FL',
 'San Juan, PR',
 'Savannah, GA Port of',
 'Seattle, WA',
 'South Je

In [42]:
p = dfP['PORT CODE'].unique().tolist() 
n = dfN['PORT CODE'].unique().tolist() 
s = dfS['PORT CODE'].unique().tolist() 


In [44]:
def chk(a,b):
    print ("A not in B:", set(a) - set(b))
    print ("B not in A:", set(b) - set(a))

chk(p,n)
chk(p,s)
chk(n,s)

A not in B: set()
B not in A: set()
A not in B: set()
B not in A: set()
A not in B: set()
B not in A: set()


In [22]:
ports_hist.head()

,Port_Name,Port ID,State,Region
0,"Alaska, AK Port of",4820,Alaska,Pacific Coast
101,"Baltimore, MD",700,Maryland,Atlantic Coast
202,"Beaumont, TX",2393,Texas,Gulf Coast & Mississippi River
281,"Boston, MA",149,Massachusetts,Atlantic Coast
382,"Cincinnati-Northern KY, Ports of",2338,Kentucky and Ohio,Gulf Coast & Mississippi River


In [77]:
#a=(df_hist['Port ID'].values.tolist())
a = vesselCalls['Port ID'].unique().tolist()

b=(df['PORT CODE'].values.tolist())

In [78]:
len(set(a)),len(set(b))

(48, 109)

In [79]:
d = list(set(a) - set(b))

In [80]:
d 

[2306, 3204, 2351, 2255, 4626, 149, 2358, 3926, 2363, 3743]

In [82]:
vesselCalls.loc[vesselCalls['Port ID'].isin(d)]['Port_Name'].unique().tolist()

['Boston, MA',
 'Kalama, WA Port of',
 'Mid-America Port Commission',
 'New Bourbon Port, MO',
 'Northern Indiana, IN',
 'Pittsburgh, PA Port of',
 'Plaquemines Port District, LA',
 'Southern Indiana Maritime District, IN',
 'Toledo-Lucas County Port, OH',
 'Two Harbors, MN']

In [83]:

xls = pd.ExcelFile("WCSC 2022 Container traffic.xlsx")
df_2022 = xls.parse("2022 TEUs by Port", header=[0, 1, 2])



In [87]:
colsNew = []
for b in df_2022.columns:
    final=""

    for c in b:
        if "Unnamed:" in c:
            continue
        else:
            final += c.strip() + " "
    colsNew.append(final.strip())

df_2022.columns = colsNew

In [88]:
df_2022.columns

Index(['PORT NAME', 'STATE', 'DOMESTIC InBound Loaded',
       'DOMESTIC InBound Empty', 'DOMESTIC OutBound Loaded',
       'DOMESTIC OutBound Empty', 'DOMESTIC Total Domestic',
       'FOREIGN InBound Loaded', 'FOREIGN OutBound Loaded',
       'FOREIGN Total Foreign Loaded', 'Grand Total Loaded'],
      dtype='object')

In [92]:
df.columns

Index(['PORT CODE', 'PORT NAME', 'STATE', 'DOMESTIC InBound Loaded',
       'DOMESTIC InBound Empty', 'DOMESTIC OutBound Loaded',
       'DOMESTIC OutBound Empty', 'DOMESTIC Total Domestic',
       'FOREIGN InBound Loaded', 'FOREIGN OutBound Loaded',
       'FOREIGN Total Foreign Loaded', 'Grand Total Loaded', 'Imports',
       'Exports', 'Empty'],
      dtype='object')

In [89]:
df_2022['Imports'] = df_2022['DOMESTIC InBound Loaded']+ df_2022['FOREIGN InBound Loaded'] 
df_2022['Exports'] = df_2022['DOMESTIC OutBound Loaded']+ df_2022['FOREIGN OutBound Loaded'] 
df_2022['Empty'] = df_2022['DOMESTIC InBound Empty']

In [105]:
c = df_2022['PORT NAME'].values.tolist()
b = df['PORT NAME'].values.tolist()
a_2021 = df_hist.loc[(df_hist['Cargo Type'] == 'CONTAINER') & (df_hist['Reporting Year'] == 2021),'Port_Name'].values.tolist()
a_all = df_hist.loc[(df_hist['Cargo Type'] == 'CONTAINER') ,'Port_Name'].values.tolist()

In [106]:
a_2021 = set(a_2021)
a_all = set(a_all)
b = set(b)
c = set(c)

In [110]:
not23 = list(a_2021 - b)

In [111]:
not22 = list(a_2021 - c)

In [112]:
len(not23), len(not22)

(18, 18)

In [119]:
bpts=df_hist.loc[df_hist['Port_Name'].isin(not23) & (df_hist['Cargo Type'] == 'CONTAINER') & (df_hist['Reporting Year'] == 2021)]

In [115]:
df_hist

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
0,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,EMPTY,Container TEUs,18.0,42.91847184,"58,059.2"
1,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,EXPORTS,Container TEUs,18.0,-5.893463975,"71,169.95"
2,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,IMPORTS,Container TEUs,18.0,-16.02295603,"205,662.3"
3,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,TOTAL,Container TEUs,18.0,-7.271842483,"334,891.45"
4,DRY BULK,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,DOMESTIC,Short Tons,161.0,-10.15539526,"96,901"
...,...,...,...,...,...,...,...,...,...,...,...
5184,VESSEL CALLS,7713,Illinois Waterway Ports and Terminals Port Sta...,NaN,2020,Illinois,Container,Vessel Calls,NaN,0,NaN
5185,VESSEL CALLS,7713,Illinois Waterway Ports and Terminals Port Sta...,NaN,2020,Illinois,Dry Bulk,Vessel Calls,NaN,0,NaN
5186,VESSEL CALLS,7713,Illinois Waterway Ports and Terminals Port Sta...,NaN,2020,Illinois,Dry Bulk Barge,Vessel Calls,NaN,0,"7,353.5"
5187,VESSEL CALLS,7713,Illinois Waterway Ports and Terminals Port Sta...,NaN,2020,Illinois,Other Freight,Vessel Calls,NaN,0,NaN


In [121]:
bpts['Port_Name'].unique().tolist()

['Alaska, AK Port of',
 'Boston, MA',
 'Cleveland-Cuyahoga Port, OH',
 'Houston Port Authority, TX',
 'Lake Charles Harbor District, LA',
 'Long Beach, CA Port of',
 'Longview, WA Port of',
 'Los Angeles, CA Port of',
 'New Orleans, LA',
 'New York, NY & NJ',
 'Oakland, CA Port of',
 'Philadelphia Regional Port, PA',
 'Plaquemines Port District, LA',
 'Savannah, GA Port of',
 'Seattle, WA',
 'South Jersey Port, NJ',
 'South Louisiana, LA, Port of',
 'Virginia, VA, Port of']

In [122]:
display(bpts.head(100))

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
70,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,EMPTY,Container TEUs,18.0,6.202991905,"130,415.15"
71,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,EXPORTS,Container TEUs,18.0,9.796983973,"101,102.6"
72,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,IMPORTS,Container TEUs,18.0,8.457956315,"314,558.55"
73,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2021,Alaska,TOTAL,Container TEUs,18.0,8.153731129,"546,076.3"
351,CONTAINER,149,"Boston, MA",Atlantic Coast,2021,Massachusetts,EMPTY,Container TEUs,25.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
3473,CONTAINER,2253,"South Louisiana, LA, Port of",Gulf Coast & Mississippi River,2021,Louisiana,TOTAL,Container TEUs,85.0,0,51.54
4137,CONTAINER,5700,"Virginia, VA, Port of",Atlantic Coast,2021,Virginia,EMPTY,Container TEUs,5.0,0,0
4138,CONTAINER,5700,"Virginia, VA, Port of",Atlantic Coast,2021,Virginia,EXPORTS,Container TEUs,5.0,19.254530383,"1,016,780.13"
4139,CONTAINER,5700,"Virginia, VA, Port of",Atlantic Coast,2021,Virginia,IMPORTS,Container TEUs,5.0,32.85021461,"1,753,184.56"


In [125]:
display(df_hist[(df_hist['Cargo Type'] == 'CONTAINER') & (df_hist['Reporting Year'] == 2021) & (df_hist['Volume'] == '0')])
#display(df_hist[(df_hist['Cargo Type'] == 'CONTAINER') & (df_hist['Reporting Year'] == 2021) ])


,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
171,CONTAINER,700,"Baltimore, MD",Atlantic Coast,2021,Maryland,EMPTY,Container TEUs,15.0,0,0
252,CONTAINER,2393,"Beaumont, TX",Gulf Coast & Mississippi River,2021,Texas,EMPTY,Container TEUs,94.0,0,0
351,CONTAINER,149,"Boston, MA",Atlantic Coast,2021,Massachusetts,EMPTY,Container TEUs,25.0,0,0
472,CONTAINER,3217,"Cleveland-Cuyahoga Port, OH",Great Lakes,2021,Ohio,EMPTY,Container TEUs,68.0,0,0
473,CONTAINER,3217,"Cleveland-Cuyahoga Port, OH",Great Lakes,2021,Ohio,EXPORTS,Container TEUs,68.0,0,0
550,CONTAINER,2436,"Corpus Christi, TX",Gulf Coast & Mississippi River,2021,Texas,EMPTY,Container TEUs,73.0,0,0
551,CONTAINER,2436,"Corpus Christi, TX",Gulf Coast & Mississippi River,2021,Texas,EXPORTS,Container TEUs,73.0,0,0
631,CONTAINER,3924,"Duluth-Superior, MN and WI",Great Lakes,2021,Minnesota and Wisconsin,EMPTY,Container TEUs,90.0,0,0
632,CONTAINER,3924,"Duluth-Superior, MN and WI",Great Lakes,2021,Minnesota and Wisconsin,EXPORTS,Container TEUs,90.0,0,0
977,CONTAINER,2031,"Houston Port Authority, TX",Gulf Coast & Mississippi River,2021,Texas,EMPTY,Container TEUs,6.0,0,0


In [127]:
sorted(not23)

['Alaska, AK Port of',
 'Boston, MA',
 'Cleveland-Cuyahoga Port, OH',
 'Houston Port Authority, TX',
 'Lake Charles Harbor District, LA',
 'Long Beach, CA Port of',
 'Longview, WA Port of',
 'Los Angeles, CA Port of',
 'New Orleans, LA',
 'New York, NY & NJ',
 'Oakland, CA Port of',
 'Philadelphia Regional Port, PA',
 'Plaquemines Port District, LA',
 'Savannah, GA Port of',
 'Seattle, WA',
 'South Jersey Port, NJ',
 'South Louisiana, LA, Port of',
 'Virginia, VA, Port of']

In [129]:
#c = df_2022['PORT NAME'].values.tolist()
b = df['PORT CODE'].values.tolist()
a_2021 = df_hist.loc[(df_hist['Cargo Type'] == 'CONTAINER') & (df_hist['Reporting Year'] == 2021),'Port ID'].values.tolist()
a_all = df_hist.loc[(df_hist['Cargo Type'] == 'CONTAINER') ,'Port ID'].values.tolist()

In [130]:
not23 = list(set(a_2021) - set(b))

In [131]:
not23

[149, 2255]

In [132]:
a_2021 = df_hist.loc[(df_hist['Cargo Type'] == 'CONTAINER') & (df_hist['Reporting Year'] == 2021) & (df_hist['Port ID'].isin(not23))]


In [133]:
a_2021

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
351,CONTAINER,149,"Boston, MA",Atlantic Coast,2021,Massachusetts,EMPTY,Container TEUs,25.0,0,0
352,CONTAINER,149,"Boston, MA",Atlantic Coast,2021,Massachusetts,EXPORTS,Container TEUs,25.0,-11.922160527,"61,172.42"
353,CONTAINER,149,"Boston, MA",Atlantic Coast,2021,Massachusetts,IMPORTS,Container TEUs,25.0,-30.325670685,"99,989.63"
354,CONTAINER,149,"Boston, MA",Atlantic Coast,2021,Massachusetts,TOTAL,Container TEUs,25.0,-24.323806406,"161,162.05"
2403,CONTAINER,2255,"Plaquemines Port District, LA",Gulf Coast & Mississippi River,2021,Louisiana,EMPTY,Container TEUs,98.0,0,0
2404,CONTAINER,2255,"Plaquemines Port District, LA",Gulf Coast & Mississippi River,2021,Louisiana,EXPORTS,Container TEUs,98.0,0,0
2405,CONTAINER,2255,"Plaquemines Port District, LA",Gulf Coast & Mississippi River,2021,Louisiana,IMPORTS,Container TEUs,98.0,0,10
2406,CONTAINER,2255,"Plaquemines Port District, LA",Gulf Coast & Mississippi River,2021,Louisiana,TOTAL,Container TEUs,98.0,0,10


In [138]:
# Filter rows where port name contains "boston" (case-insensitive)
#df[df['PORT NAME'].str.contains('boston', case=False, na=False)]
df[df['PORT NAME'].str.contains('plaquemines', case=False, na=False)]



,PORT CODE,PORT NAME,STATE,DOMESTIC InBound Loaded,DOMESTIC InBound Empty,DOMESTIC OutBound Loaded,DOMESTIC OutBound Empty,DOMESTIC Total Domestic,FOREIGN InBound Loaded,FOREIGN OutBound Loaded,FOREIGN Total Foreign Loaded,Grand Total Loaded,Imports,Exports,Empty


In [144]:
#display(sorted(df['PORT NAME'].unique()[:140]))
display(sorted(df_2022['PORT NAME'].unique()[:140]))

['Baltimore, MD',
 'Beaumont, TX',
 'Bellingham, WA',
 'Bethel, AK',
 'Brevig Mission, AK',
 'Buffalo, NY',
 'Calhoun Port Authority, TX',
 'Canaveral Port District, FL',
 'Cheboygan, MI',
 'Clallam County Port District, WA',
 'Cleveland-Cuyahoga County, OH',
 'Cordova, AK',
 'Corpus Christi, TX',
 'DeTour, MI',
 'Detroit/Wayne County Port Authority, MI',
 'Dillingham, AK',
 'Duluth-Superior, MN and WI',
 'Fall River Port Authority, MA',
 'False Pass, AK',
 'Galveston, TX',
 'Guayanilla, PR',
 'Guaynabo, PR',
 'Haines Borough, AK',
 "Hilo, Hawai'i, HI",
 "Honolulu, O'ahu, HI",
 'Hoonah, AK',
 'Hooper Bay, AK',
 'Illinois International Port District, IL',
 'Jacksonville, FL',
 'Jefferson County Port District, WA',
 'Juneau, AK',
 'Kahului, Maui, HI',
 'Kake, AK',
 "Kaumalapau, Lana'i, HI",
 "Kaunakakai, Moloka'i, HI",
 "Kawaihae, Hawai'i, HI",
 'Ketchikan, AK',
 'Kivalina, AK',
 'Kodiak, AK',
 'Lake Charles Harbor and Terminal District, LA',
 'Manatee County Port Authority, FL',
 'Metla

### Extra

In [78]:
dfPDE = pd.read_csv("PortDataExport.csv")

In [83]:
pdeCont=dfPDE.loc[dfPDE['Cargo Type'] == "CONTAINER"]

In [86]:
pdeCont.loc[pdeCont['Reporting Year'] == 2022].head()

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
1136,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2022,New Jersey and New York,EMPTY,Container TEUs,1.0,244.655929722,"9,416"
1137,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2022,New Jersey and New York,EXPORTS,Container TEUs,1.0,-4.050814244,"1,280,126"
1138,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2022,New Jersey and New York,IMPORTS,Container TEUs,1.0,8.255370536,"5,380,191"
1139,CONTAINER,398,"New York, NY & NJ",Atlantic Coast,2022,New Jersey and New York,TOTAL,Container TEUs,1.0,5.650936820,"6,660,317"
1148,CONTAINER,550,"South Jersey Port, NJ",Atlantic Coast,2022,New Jersey,EMPTY,Container TEUs,24.0,-3.616872467,"23,565"


In [85]:
df.head()

,PORT NAME,STATE,DOMESTIC InBound Loaded,DOMESTIC InBound Empty,DOMESTIC OutBound Loaded,DOMESTIC OutBound Empty,DOMESTIC Total Domestic,FOREIGN InBound Loaded,FOREIGN OutBound Loaded,FOREIGN Total Foreign Loaded,Grand Total Loaded
0,"Port Authority of New York and New Jersey, NY ...","NY,NJ",28057.0,4707.75,28061.25,4707.75,65533.75,5352133.91,1252064.92,6604198.83,6660317.08
1,"Port of Los Angeles, CA",CA,0.0,0.00,0.00,0.00,0.00,5231585.42,1192756.02,6424341.44,6424341.44
2,"Port of Long Beach, CA",CA,39742.4,19589.20,319651.10,2001.00,380983.70,4675674.03,1056909.71,5732583.74,6091977.24
3,"Port of Savannah, GA",GA,0.0,0.00,0.00,0.00,0.00,3012685.25,1317230.12,4329915.37,4329915.37
4,"Port of Houston Authority of Harris County, TX",TX,804.0,0.00,0.00,567.00,1371.00,2034773.64,1216978.92,3251752.56,3252556.56


In [88]:
a=df.loc[df['PORT NAME'] == "Port Authority of New York and New Jersey, NY & NJ"]

In [100]:
a['Imports'] = a['DOMESTIC InBound Loaded']+ a['FOREIGN InBound Loaded'] 
a['Exports'] = a['DOMESTIC OutBound Loaded']+ a['FOREIGN OutBound Loaded'] 
a['Empty'] = a['DOMESTIC InBound Empty']
print(a)

                                           PORT NAME  STATE  \
0  Port Authority of New York and New Jersey, NY ...  NY,NJ   

   DOMESTIC InBound Loaded  DOMESTIC InBound Empty  DOMESTIC OutBound Loaded  \
0                  28057.0                 4707.75                  28061.25   

   DOMESTIC OutBound Empty  DOMESTIC Total Domestic  FOREIGN InBound Loaded  \
0                  4707.75                 65533.75              5352133.91   

   FOREIGN OutBound Loaded  FOREIGN Total Foreign Loaded  Grand Total Loaded  \
0               1252064.92                    6604198.83          6660317.08   

      Imports     Exports    Empty  
0  5380190.91  1280126.17  4707.75  


/tmp/ipykernel_118122/3886418078.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  a['Imports'] = a['DOMESTIC InBound Loaded']+ a['FOREIGN InBound Loaded']
/tmp/ipykernel_118122/3886418078.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  a['Exports'] = a['DOMESTIC OutBound Loaded']+ a['FOREIGN OutBound Loaded']
/tmp/ipykernel_118122/3886418078.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

In [94]:
pdeCont.loc[(pdeCont['Port_Name'] == "New York, NY & NJ") & (pdeCont['Reporting Year'] == 2022),['Trade Type','Volume'] ]

,Trade Type,Volume
1136,EMPTY,"9,416"
1137,EXPORTS,"1,280,126"
1138,IMPORTS,"5,380,191"
1139,TOTAL,"6,660,317"


In [ ]:
Domestic Inbound Loaded
Domestic Inbound Empty
Domestic Outbound Loaded
Domestic Outbound Empty
Foreign Inbound Loaded
Foreign Outbound Loaded

In [43]:
a=df_hist[df_hist['Cargo Type'] == 'CONTAINER']

In [44]:
a.head()

,Cargo Type,Port ID,Port_Name,Region,Reporting Year,State,Trade Type,Units,Port Ranking,Percent Change,Volume
0,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,EMPTY,Container TEUs,18.0,42.9,"58,059.2"
1,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,EXPORTS,Container TEUs,18.0,-5.9,"71,169.95"
2,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,IMPORTS,Container TEUs,18.0,-16.0,"205,662.3"
3,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2016,Alaska,TOTAL,Container TEUs,18.0,-7.3,"334,891.45"
14,CONTAINER,4820,"Alaska, AK Port of",Pacific Coast,2017,Alaska,EMPTY,Container TEUs,19.0,15.1,"66,817.1"


In [45]:
a['Port_Name'].value_counts().sort_index()

Port_Name
Alaska, AK Port of                  24
Baltimore, MD                       24
Beaumont, TX                         4
Boston, MA                          24
Cleveland-Cuyahoga Port, OH          4
Corpus Christi, TX                   4
Duluth-Superior, MN and WI           4
Gulfport, MS                        20
Honolulu, O'ahu, HI                 24
Houston Port Authority, TX          24
Jacksonville, FL                    24
Lake Charles Harbor District, LA     4
Long Beach, CA Port of              24
Longview, WA Port of                 4
Los Angeles, CA Port of             24
Mobile, AL                          24
New Orleans, LA                     24
New York, NY & NJ                   24
Oakland, CA Port of                 24
Philadelphia Regional Port, PA      24
Plaquemines Port District, LA        4
Port Arthur, TX                      4
Port Everglades, FL                 24
Port Freeport, TX                   24
Port of Charleston, SC              24
Port of Palm Be